# Digit Recognition — MNIST Sequence

**Computer Vision Project** — Classify handwritten digits 0–9.

Model: PyTorch CNN  
Dataset: MNIST (60K train, 10K test, 28×28 grayscale)  
Target: digit class (0–9)

## 2 · Project Overview

We build a **Convolutional Neural Network (CNN)** to recognize handwritten digits from the MNIST dataset. MNIST is the "Hello World" of computer vision — 70,000 grayscale images of digits 0–9, each 28×28 pixels. This project teaches CNN fundamentals: convolution, pooling, batch normalization, and training loops.

## 3 · Learning Objectives

By the end of this notebook you will be able to:
1. Load and preprocess image datasets using torchvision.
2. Build a CNN architecture with Conv2d, BatchNorm, MaxPool, and Dropout.
3. Train a PyTorch model with proper train/validation loops.
4. Evaluate with accuracy, F1, and confusion matrix.
5. Visualize predictions and misclassifications.

## 4 · Problem Statement

Given a 28×28 grayscale image of a handwritten digit, classify it into one of 10 classes (0–9). This is a **multi-class image classification** task.

## 5 · Why This Project Matters

- MNIST is the standard benchmark for learning CNNs.
- Handwriting recognition is used in postal automation, check processing, and form digitization.
- CNN fundamentals transfer to all image classification tasks.
- Understanding training dynamics (loss curves, overfitting) is essential for deep learning.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Train samples** | 60,000 |
| **Test samples** | 10,000 |
| **Image size** | 28 × 28 × 1 (grayscale) |
| **Classes** | 10 (digits 0–9) |
| **Source** | torchvision built-in |

## 7 · Dataset Source and License Notes

- **Source**: Yann LeCun's MNIST database, available via `torchvision.datasets.MNIST`.
- **License**: Public domain / research use.
- **Auto-download**: torchvision downloads it automatically on first use.

## 8 · Environment Setup

In [ ]:
import subprocess, sys

def _install(pkg, imp=None):
    try: __import__(imp or pkg)
    except ImportError: subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

_install("torch")
_install("torchvision")
print("All packages ready.")

## 9 · Imports

In [ ]:
import warnings, os, json
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

from sklearn.metrics import (accuracy_score, f1_score, confusion_matrix,
                             classification_report, ConfusionMatrixDisplay)

SAVE_DIR = os.path.dirname(os.path.abspath("__file__"))
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## 10 · Configuration / Constants

In [ ]:
BATCH_SIZE = 128
EPOCHS = 10
LR = 0.001
NUM_CLASSES = 10
print(f"Batch size: {BATCH_SIZE}, Epochs: {EPOCHS}, LR: {LR}")

## 11 · Dataset Download or Loading

MNIST is downloaded automatically by torchvision.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

full_train = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_ds = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

# Split train into train/val
train_size = int(0.85 * len(full_train))
val_size = len(full_train) - train_size
train_ds, val_ds = random_split(full_train, [train_size, val_size],
                                 generator=torch.Generator().manual_seed(SEED))

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}")

## 12 · Data Validation Checks

In [ ]:
batch_x, batch_y = next(iter(train_loader))
print(f"Batch shape: {batch_x.shape}")
print(f"Label shape: {batch_y.shape}")
print(f"Pixel range: [{batch_x.min():.2f}, {batch_x.max():.2f}]")
print(f"Unique labels in batch: {sorted(batch_y.unique().tolist())}")

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i in range(16):
    ax = axes[i // 8, i % 8]
    ax.imshow(batch_x[i, 0].numpy(), cmap="gray")
    ax.set_title(f"{batch_y[i].item()}")
    ax.axis("off")
plt.suptitle("Sample Training Images")
plt.tight_layout()
plt.show()

## 13 · Exploratory Data Analysis

We check the class distribution across the training set.

In [ ]:
all_labels = []
for _, labels in train_loader:
    all_labels.extend(labels.tolist())

from collections import Counter
label_counts = Counter(all_labels)
classes = sorted(label_counts.keys())
counts = [label_counts[c] for c in classes]

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(classes, counts, color="steelblue")
ax.set_xlabel("Digit")
ax.set_ylabel("Count")
ax.set_title("Training Set Class Distribution")
ax.set_xticks(classes)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "eda_class_distribution.png"), dpi=100)
plt.show()
print(f"Class counts: {dict(sorted(label_counts.items()))}")

## 14 · Target Analysis

MNIST is nearly balanced across all 10 digit classes, so no special class weighting is needed.

## 15 · Train/Validation/Test Split Strategy

We use the standard MNIST train/test split (60K/10K) with an 85/15 train/val split from the training set.

## 16 · Preprocessing Strategy

Images are normalized with MNIST-specific mean (0.1307) and std (0.3081). No augmentation is used for this baseline.

## 17 · Feature Engineering

For CNNs, the raw pixel values (after normalization) serve as features. The convolutional layers automatically learn hierarchical feature representations.

## 18 · Baseline Model

Our CNN architecture:

In [ ]:
class MNISTNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, NUM_CLASSES),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

model = MNISTNet().to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total_params:,}")
print(model)

## 19 · LazyPredict Benchmark

**Not applicable for computer vision / CNN tasks.** LazyPredict is designed for tabular classification and regression only.

## 20 · FLAML AutoML Run

**Not applicable for computer vision / CNN tasks.** FLAML is designed for tabular AutoML.

## 21 · Training

We train the CNN with Adam optimizer and CrossEntropyLoss.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

train_losses, val_losses = [], []
train_accs, val_accs = [], []

for epoch in range(EPOCHS):
    # Train
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * X_batch.size(0)
        correct += (outputs.argmax(1) == y_batch).sum().item()
        total += X_batch.size(0)
    train_losses.append(running_loss / total)
    train_accs.append(correct / total)

    # Validate
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            val_loss += loss.item() * X_batch.size(0)
            val_correct += (outputs.argmax(1) == y_batch).sum().item()
            val_total += X_batch.size(0)
    val_losses.append(val_loss / val_total)
    val_accs.append(val_correct / val_total)

    print(f"Epoch {epoch+1}/{EPOCHS} — Train Loss: {train_losses[-1]:.4f} Acc: {train_accs[-1]:.4f} | Val Loss: {val_losses[-1]:.4f} Acc: {val_accs[-1]:.4f}")

## 22 · Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(train_losses, label="Train Loss")
ax1.plot(val_losses, label="Val Loss")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_title("Loss Curves")
ax1.legend()

ax2.plot(train_accs, label="Train Acc")
ax2.plot(val_accs, label="Val Acc")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy")
ax2.set_title("Accuracy Curves")
ax2.legend()

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "training_curves.png"), dpi=100)
plt.show()

## 23 · Final Evaluation on Test Set

In [ ]:
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        outputs = model(X_batch)
        all_preds.extend(outputs.argmax(1).cpu().tolist())
        all_labels.extend(y_batch.tolist())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

acc = accuracy_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds, average="weighted")
print(f"Test Accuracy: {acc:.4f}")
print(f"Test F1 (weighted): {f1:.4f}")

## 24 · Error Analysis

In [ ]:
cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(10, 8))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=list(range(10)))
disp.plot(ax=ax, cmap="Blues", values_format="d")
ax.set_title("Confusion Matrix — MNIST CNN")
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "confusion_matrix.png"), dpi=100)
plt.show()

print(classification_report(all_labels, all_preds))

# Show misclassified examples
wrong = np.where(all_preds != all_labels)[0]
print(f"\nTotal errors: {len(wrong)} / {len(all_labels)} ({len(wrong)/len(all_labels)*100:.2f}%)")

if len(wrong) > 0:
    fig, axes = plt.subplots(2, 5, figsize=(14, 6))
    for i, idx in enumerate(wrong[:10]):
        ax = axes[i // 5, i % 5]
        img, _ = test_ds[idx]
        ax.imshow(img[0].numpy(), cmap="gray")
        ax.set_title(f"True={all_labels[idx]} Pred={all_preds[idx]}")
        ax.axis("off")
    plt.suptitle("Sample Misclassifications")
    plt.tight_layout()
    plt.show()

## 25 · Interpretation and Business Insight

- The CNN achieves **>99%** accuracy on MNIST, which is expected for a well-tuned CNN.
- Most confusion is between visually similar digits: 4↔9, 3↔5, 7↔2.
- For production digit recognition, the model would need to handle varying image quality, rotation, and noise.

## 26 · Limitations

- MNIST is too easy — most real-world OCR tasks are much harder.
- Clean, centered, normalized images don't reflect real-world handwriting.
- No data augmentation was used.

## 27 · How to Improve This Project

1. Add data augmentation (rotation, scaling, elastic distortion).
2. Try deeper architectures (ResNet-style) or pretrained models.
3. Test on harder datasets: EMNIST, SVHN, or real document images.
4. Add learning rate scheduling (CosineAnnealing).

## 28 · Production Considerations

- Model quantization/ONNX export for deployment.
- Preprocessing pipeline must match training normalization.
- Confidence thresholding — reject low-confidence predictions.
- Handle out-of-distribution inputs (non-digit images).

## 29 · Common Mistakes

1. Not normalizing images with dataset-specific statistics.
2. No validation set — overfitting goes undetected.
3. Using fully-connected layers only (ignoring spatial structure).
4. Not using Dropout/BatchNorm — leads to overfitting on small datasets.

## 30 · Mini Challenge / Exercises

1. Add data augmentation (RandomRotation, RandomAffine) — does it improve?
2. Replace AdaptiveAvgPool with flatten + larger FC layers — compare parameter count and accuracy.
3. Try training for 20 epochs with CosineAnnealingLR.

## 31 · Final Summary / Key Takeaways

In [ ]:
results = {
    "CNN": {"accuracy": acc, "f1": f1}
}
metrics = {
    "best_model": "CNN",
    "best_accuracy": acc,
    "best_f1": f1,
    "epochs": EPOCHS,
    "total_parameters": sum(p.numel() for p in model.parameters()),
}

# Save model
torch.save(model.state_dict(), os.path.join(SAVE_DIR, "best_model.pth"))
print("Model saved to best_model.pth")

with open(os.path.join(SAVE_DIR, "metrics.json"), "w") as f:
    json.dump(metrics, f, indent=2)
print("Metrics saved to metrics.json")
print(json.dumps(metrics, indent=2))

print("\n✅ Digit Recognition — MNIST Sequence notebook complete.")